# 02 — Model Audit & Inference

**Purpose:** Run inference on the **validation set** for all 20 trained models, then generate:
1. `val_results_*.json` files (COCO-format predictions for WBF fusion)
2. Validation leaderboard ranked by Absolute Score

In [1]:
import os, gc, json
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from mmengine.config import Config
from mmdet.apis import init_detector, inference_detector

# --- CONFIGURATION ---
DATA_ROOT = r"C:\Users\dadab\Desktop\Clean Version\data"
MODELS_DIR = r"C:\Users\dadab\Desktop\Clean Version\models"
CONFIG_PATH = r"C:\Users\dadab\Desktop\Clean Version\mask-rcnn_r50_fpn_1x_coco.py"
OUTPUT_DIR = "outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VAL_IMG_DIR = os.path.join(DATA_ROOT, "val/images")
VAL_JSON = os.path.join(DATA_ROOT, "val/val_annotations.json")

IOU_THRES = 0.50
CONF_THRES = 0.50

## Utility Functions

In [2]:
def get_box_iou(boxA, boxB):
    """IoU for [x1, y1, x2, y2] format."""
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0:
        return 0.0
    union = (boxA[2]-boxA[0])*(boxA[3]-boxA[1]) + (boxB[2]-boxB[0])*(boxB[3]-boxB[1]) - inter
    return inter / union

def load_model(config_path, checkpoint_path):
    cfg = Config.fromfile(config_path)
    cfg.model.roi_head.bbox_head.num_classes = 2
    cfg.model.roi_head.mask_head.num_classes = 2
    return init_detector(cfg, checkpoint_path, device='cuda:0')

## Run Inference + Forensic Audit

In [3]:
print("Loading validation ground truth...")
with open(VAL_JSON, 'r') as f:
    coco = json.load(f)

ann_map = {}
for ann in coco['annotations']:
    x, y, w, h = ann['bbox']
    ann_map.setdefault(ann['image_id'], []).append([x, y, x+w, y+h])
img_map = {img['id']: img['file_name'] for img in coco['images']}

model_files = sorted([f for f in os.listdir(MODELS_DIR) if f.endswith(".pth")])
print(f"Found {len(model_files)} models")

model_stats = []
for model_name in model_files:
    print(f"\nProcessing: {model_name}")
    model = load_model(CONFIG_PATH, os.path.join(MODELS_DIR, model_name))
    coco_preds, total_found, total_fp = [], 0, 0

    for img_id, fname in tqdm(img_map.items(), desc="Inference"):
        img_path = os.path.join(VAL_IMG_DIR, fname)
        if not os.path.exists(img_path):
            continue
        result = inference_detector(model, img_path)
        pred = result.pred_instances
        valid = pred.scores > 0.01
        p_boxes = []

        for i in range(valid.sum()):
            box = pred.bboxes[valid][i].cpu().numpy()
            score = float(pred.scores[valid][i])
            label = int(pred.labels[valid][i]) + 1
            coco_preds.append({
                "image_id": img_id, "category_id": label,
                "bbox": [float(box[0]), float(box[1]),
                         float(box[2]-box[0]), float(box[3]-box[1])],
                "score": score, "segmentation": []
            })
            if score >= CONF_THRES:
                p_boxes.append(box)

        # Forensic audit
        gts = ann_map.get(img_id, [])
        matched = set()
        for p_box in p_boxes:
            hit = False
            for j, g_box in enumerate(gts):
                if j not in matched and get_box_iou(p_box, g_box) >= IOU_THRES:
                    matched.add(j); total_found += 1; hit = True; break
            if not hit:
                total_fp += 1

    # Save predictions
    out_name = f"val_results_{model_name.replace('.pth', '')}.json"
    with open(os.path.join(OUTPUT_DIR, out_name), 'w') as f:
        json.dump(coco_preds, f)

    model_stats.append({
        "Model": model_name.replace(".pth", ""),
        "Found": total_found, "FP": total_fp,
        "Score": total_found - (total_fp * 0.1)
    })
    del model; torch.cuda.empty_cache(); gc.collect()

# Leaderboard
df = pd.DataFrame(model_stats).sort_values("Score", ascending=False).reset_index(drop=True)
print("\n=== VALIDATION LEADERBOARD ===")
print(df.to_string())
df.to_csv(os.path.join(OUTPUT_DIR, "val_leaderboard.csv"), index=False)

Loading validation ground truth...
Found 20 models

Processing: exp01_baseline_best.pth
Loads checkpoint by local backend from path: C:\Users\dadab\Desktop\Clean Version\models\exp01_baseline_best.pth


Inference: 100%|█████████████████████████████████████████████████████████████████████| 604/604 [00:22<00:00, 26.72it/s]



Processing: exp02_hflip_best.pth
Loads checkpoint by local backend from path: C:\Users\dadab\Desktop\Clean Version\models\exp02_hflip_best.pth


Inference: 100%|█████████████████████████████████████████████████████████████████████| 604/604 [00:21<00:00, 27.69it/s]



Processing: exp03_vflip_best.pth
Loads checkpoint by local backend from path: C:\Users\dadab\Desktop\Clean Version\models\exp03_vflip_best.pth


...


Processing: exp19_gaussianblur_best.pth
Loads checkpoint by local backend from path: C:\Users\dadab\Desktop\Clean Version\models\exp19_gaussianblur_best.pth


Inference: 100%|█████████████████████████████████████████████████████████████████████| 604/604 [00:20<00:00, 28.80it/s]



Processing: exp20_medianblur_best.pth
Loads checkpoint by local backend from path: C:\Users\dadab\Desktop\Clean Version\models\exp20_medianblur_best.pth


Inference: 100%|█████████████████████████████████████████████████████████████████████| 604/604 [00:21<00:00, 28.49it/s]



=== VALIDATION LEADERBOARD ===
                                 Model  Found   FP   Score
0                     exp02_hflip_best   1558  614  1496.6
1                    exp04_rotate_best   1552  598  1492.2
2                     exp03_vflip_best   1559  669  1492.1
3                       exp09_hsv_best   1535  523  1482.7
4                    exp06_affine_best   1540  583  1481.7
5                  exp01_baseline_best   1536  585  1477.5
6   exp07_randomresize_randomcrop_best   1548  725  1475.5
7            exp12_griddistortion_best   1530  609  1469.1
8                exp18_motionblur_best   1522  538  1468.2
9               exp16_griddropout_best   1517  549  1462.1
10                    exp05_scale_best   1513  511  1461.9
11               exp20_medianblur_best   1515  558  1459.2
12       exp08_brightnesscontrast_best   1509  524  1456.6
13               exp17_gaussnoise_best   1507  521  1454.9
14        exp13_opticaldistortion_best   1507  523  1454.7
15                    ex